In [1]:
%load_ext autoreload
%autoreload 2

import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd
from functions import *

import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)

import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph1 import DFGrepInterferencePartitionBased, DFGrepBurstiness, DFGrepWorkflow, DFGrepWorkflow1

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None

# App Name
app_name = "montage_pegasus-dss-1deg_node-16"

cp_dir = "/p/lustre3/pandey2/logs/Results_Checkpoint/dataflow/"+app_name+"/"
os.makedirs(cp_dir, exist_ok=True)

condition_fn = None #

if app_name == "montage_pegasus-dss-1deg_node-16":
    filename = "/p/lustre3/pandey2/logs/RAW_copy/montage_pegasus-dss-1deg_node-16/COMPACT/*.pfw.gz"

else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, debug=True,
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=True, 
                                host_pattern=r'lassen(\d+)', time_granularity=1e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()


def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))

def cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return '/'.join(path.split('/', 3)[:3])

    if "M" == json_object["ph"] and "FH" == json_object["name"] and "args" in json_object and "name" in json_object["args"]:
        d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=json_object["args"]["name"])
    if "args" in json_object and "M" != json_object["ph"]:
        if "ret" in json_object["args"]:
            d["size"] = int(json_object["args"]["ret"]) 
    if "args" in json_object:
        if "exec_hash" in json_object["args"]:
            d["exec_hash"] = json_object["args"]["exec_hash"]

    if "name" in json_object:
        if (json_object["name"] in ["fwrite", "write","pwrite","fputs"]):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1
    return d

load_cols = {'size': "int64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]", 'exec_hash':"string[pyarrow]" }
load_cols_metadata = {"FH":{'mount_point':"string[pyarrow]" }}

analyzer = DFAnalyzer(filename,load_fn=cols_function, load_cols=load_cols, load_data={"mount_point":trie}, metadata_cols = load_cols_metadata)
# analyzer = DFAnalyzer(filename)


/usr/workspace/pandey2/DFT/envDFT/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
[INFO] [11:09:46] Initialized Client with 576 workers and link http://134.9.71.27:8787/status [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:769]


/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/__init__.py


[INFO] [11:09:57] Restarting all workers [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:761]
[INFO] [11:09:59] Created index for 14 files [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:428]
[INFO] [11:09:59] Total size of all files are <dask.bag.core.Item object at 0x1555416f0100> bytes [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:430]
[INFO] [11:09:59] test debug [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:433]
[INFO] [11:10:01] Loading 2013 batches out of 14 files and has 32884998 lines overall [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:444]
[INFO] [11:10:58] Loaded events [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:511]
[INFO] [11:10:58] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFT/dftracer/dfanalyzer/main.py:517]


In [2]:
#IMP1
start_events = analyzer.events.query("name == 'start'")
sh =  analyzer.string_hash.reset_index()
# start_events.head()

# Make sure both columns are same dtype
start_events = start_events.assign(exec_hash=start_events["exec_hash"].astype("string"))
sh = sh.assign(hash=sh["hash"].astype("string"))

# Select and rename first (keeps it as a Dask DataFrame)
sh_subset = sh[["hash", "name"]].rename(columns={"name": "app name"})

# Now perform merge safely
start_events = start_events.merge(sh_subset, left_on="exec_hash", right_on="hash", how="left")


prod_cons_events = analyzer.events.query("name in ['read', 'fgets', 'fread', 'fwrite', 'write', 'fputs']")
# prod_cons_events = analyzer.events

# grouped_prod_cons = (
#     prod_cons_events.groupby(['name','cat','pid','tid','hhash','fhash','prod','cons'])[["size"]]
#     .sum()
#     .reset_index()
#     .rename(columns={'size': 'total_size'})
# ).query('total_size > 0')

grouped_prod_cons = (
    prod_cons_events.groupby(['name','cat','pid','tid','hhash','fhash','prod','cons'])[["size","dur"]]
    .sum()
    .reset_index()
    .rename(columns={'size': 'total_size'})
)

# grouped_prod_cons = (
#     prod_cons_events
#     .groupby(['name','cat','pid','tid','hhash','fhash','prod','cons'], group_keys=False)
#     .apply(lambda df: pd.Series({
#         'total_size': df['size'].sum(),
#         'total_duration': df['dur'].sum(),
#         'total_dur': merge_and_sum_durations(df)
#     }))
#     .reset_index()
# )

merged = grouped_prod_cons.merge(start_events[["pid","tid","app name"]], on=["pid","tid"], how="left")





In [3]:
all_tasks = merged['app name'].unique().compute().tolist()
all_tasks
# hosts = analyzer.events.hhash.unique().compute().tolist()
# analyzer.events1 = analyzer.events[analyzer.events["hhash"].isin(hosts)].copy()

['pegasus-mpi-cluster',
 'mFitplane',
 'pegasus-kickstart',
 'mImgtbl',
 'mViewer',
 'mDiff',
 'mBackground',
 'mConcatFit',
 'mProject',
 'mAdd',
 'flux-job',
 'mBgModel']

In [ ]:
list_a = ['mImgtbl',
 'mDiff',
 'mBackground',
 'mAdd',
 'mBgModel']

list_b = ['mFitplane',
 'mImgtbl',
 'mViewer',
 'mDiff',
 'mBackground',
 'mConcatFit',
 'mProject',
 'mAdd',
 'mBgModel']

merged = merged[merged["app name"].isin(list_a)]
# merged.groupby('app name').count().compute()

In [4]:
def create_consistant_hash(analyzer , merged):
    fhash_dd = analyzer.file_hash.reset_index()[["hash","name"]]
    return unify_fhash_dask(fhash_dd, merged)

def build_io_graph(df):
    G = nx.DiGraph()
    for _, row in df.iterrows():
        app = str(row["app name"])
        fhash = str(row["fhash"])
        weight = float(row.get("total_size", 1))
        total_dur = float(row.get("dur", 1))

        if row["prod"] == 0: #if produced = 0 than app consumes the file (File -> App)
            # file → app
            G.add_edge(fhash, app, weight=weight, dur=total_dur)
        elif row["cons"] == 0: #if consumed = 0 than app 
            # app → file
            G.add_edge(app, fhash, weight=weight, dur = total_dur)
    return G
def update_weight(G):
    for u, v, d in G.edges(data=True):
        d["inv_size"] = 1.0 / d["weight"] if d.get("weight", 0) > 0 else 1e6
    return G


In [5]:
import time
import threading
from dask.distributed import Client
import networkx as nx

client = Client(dask_scheduler)   # or use your existing connected client

# Set this manually for the current run

peak_total_mem = 0
peak_worker_mem = {}
samples = []
monitoring = True

def monitor_worker_memory(interval=0.5):
    global peak_total_mem, peak_worker_mem, monitoring, samples

    while monitoring:
        info = client.scheduler_info()
        total_mem = 0

        row = {"time": time.time()}
        for addr, w in info["workers"].items():
            mem = w["metrics"]["memory"]   # bytes
            total_mem += mem
            peak_worker_mem[addr] = max(peak_worker_mem.get(addr, 0), mem)
            row[addr] = mem

        peak_total_mem = max(peak_total_mem, total_mem)
        row["total_mem"] = total_mem
        samples.append(row)

        time.sleep(interval)

# start background monitor
t = threading.Thread(target=monitor_worker_memory, daemon=True)
t.start()

# run your actual function
start = time.time()

fhash_df , merged_new, mapping  = create_consistant_hash(analyzer,merged)
G = build_io_graph(merged_new)
G = update_weight(G)
bc = nx.betweenness_centrality(G, weight='inv_size', normalized=True) #computing based on size of datatransfer (high transfer given more importance)
runtime = time.time() - start

# stop monitor
monitoring = False
t.join()

# metrics

peak_total_mem_gb = peak_total_mem / 1e9
max_worker_mem_gb = max(peak_worker_mem.values()) / 1e9 if peak_worker_mem else 0


print("=" * 60)
print(f"Runtime (s):              {runtime:.2f}")
print(f"Peak total memory:        {peak_total_mem_gb:.2f} GB")
print(f"Peak max worker memory:   {max_worker_mem_gb:.2f} GB")

print("\nPeak memory per worker:")
for addr, mem in peak_worker_mem.items():
    print(f"  {addr:25s} {mem/1e9:8.2f} GB")
print("=" * 60)

Runtime (s):              108.49
Peak total memory:        158.02 GB
Peak max worker memory:   0.62 GB

Peak memory per worker:
  tcp://192.168.128.171:32783     0.26 GB
  tcp://192.168.128.171:32913     0.47 GB
  tcp://192.168.128.171:32915     0.26 GB
  tcp://192.168.128.171:33193     0.25 GB
  tcp://192.168.128.171:33299     0.25 GB
  tcp://192.168.128.171:33975     0.26 GB
  tcp://192.168.128.171:34189     0.26 GB
  tcp://192.168.128.171:34809     0.25 GB
  tcp://192.168.128.171:34877     0.25 GB
  tcp://192.168.128.171:35175     0.26 GB
  tcp://192.168.128.171:35381     0.25 GB
  tcp://192.168.128.171:35593     0.25 GB
  tcp://192.168.128.171:35763     0.26 GB
  tcp://192.168.128.171:36133     0.25 GB
  tcp://192.168.128.171:36421     0.25 GB
  tcp://192.168.128.171:36495     0.25 GB
  tcp://192.168.128.171:36511     0.25 GB
  tcp://192.168.128.171:36603     0.26 GB
  tcp://192.168.128.171:36631     0.26 GB
  tcp://192.168.128.171:36637     0.25 GB
  tcp://192.168.128.171:36751   

In [6]:
print(G.number_of_nodes(), G.number_of_edges())

159 315
